# Intro to Cognee — building a GTM Brain

This notebook builds a **company brain** from Slack + Granola (sections 1–6),
then **merges in the GTM brain** — accounts, sponsors, speakers, deals, signals
and calendar — so conversations and CRM-style data live in *one* graph
(section 7). The capstone question can only be answered from the merged graph.


## The API (Cognee 1.x)

The modern surface is three async functions:

```python
await cognee.remember(text, dataset_name=...)   # 1. ingest + build the graph
await cognee.recall(query)                       # 2. ask questions
await cognee.improve(dataset=...)                # 3. refine the graph from feedback/use
```

**`remember()` replaces the old two-step `add()` + `cognify()`** — it ingests the
text, extracts entities/relationships into the graph, *and* (by default) runs a
self-improvement pass. You rarely call `cognify()` directly anymore.

> **Where did `cognify` go?** It still exists. `remember()` = `add()` +
> `cognify()` + self-improvement rolled into one. Reach for the lower-level
> `cognee.add()` + `cognee.cognify(...)` only when you want to **decouple
> ingestion from graph-building** — e.g. bulk-load hundreds of docs and build
> the graph once, re-process existing data with a different schema without
> re-uploading, or run the build as a background job. For everyday use,
> `remember()` is the one call you need.

**The improvement piece.** A company brain is never "done" — new conversations
arrive and earlier answers get corrected. Cognee bakes this in:
`remember(..., self_improvement=True)` (the default) refines the graph on every
write, `cognee.improve()` runs an explicit enrichment pass, and
`recall(..., feedback_influence=...)` lets prior feedback steer future answers.
That's what makes it a *living* graph rather than a one-shot index.

## Setup

You need an LLM key. Put `LLM_API_KEY=sk-...` in the repo's `.env` (Cognee uses
it for extraction and recall). We turn session caching **on** (`CACHING=true`)
because the feedback section (5) needs session memory.

> Run once: `uv pip install -e ".[dev]" jupyterlab ipykernel`

In [ ]:
import os, re
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd().parent / ".env")   # picks up LLM_API_KEY
os.environ.setdefault("CACHING", "true")  # session memory ON (needed for the feedback section)

import cognee

# sample_data lives at the repo root; handle running from tutorial/ or root
ROOT = Path.cwd()
DATA = ROOT / "sample_data" if (ROOT / "sample_data").exists() else ROOT.parent / "sample_data"
DATASET = "gtm_brain"

async def reset():
    """Wipe the graph so each section starts from a clean slate."""
    await cognee.prune.prune_data()
    await cognee.prune.prune_system(metadata=True)

import re as _re  # (re already imported above)
def first_speaker(text):
    m = _re.search(r"\[([^,\]]+@[^,\]]+),", text)
    return m.group(1) if m else "unknown"

def answer(results):
    """Pull the natural-language answer out of a recall() result list.
    recall() can return a graph entry (.text), a QA entry (.answer), or a
    context entry (.content) depending on how the query auto-routes."""
    for r in results:
        for attr in ("text", "answer", "content"):
            value = getattr(r, attr, None)
            if value:
                return value
    return str(results[0]) if results else "<no answer>"


async def show_graph(filename, height=500):
    """Render the current graph to its own HTML file and embed it."""
    await cognee.visualize_graph(str(ROOT / filename))
    from IPython.display import IFrame
    return IFrame(src=filename, width="100%", height=height)


from cognee.api.v1.session import add_feedback, get_session

async def show_session(session_id):
    """Print what's stored in session memory for each Q&A turn."""
    entries = await get_session(session_id)
    if not entries:
        print("  (session empty)"); return
    for e in entries:
        print(f"  Q: {(e.question or '')[:55]}")
        print(f"     feedback_text : {e.feedback_text}")
        print(f"     feedback_score: {e.feedback_score}")
        print(f"     memify_metadata: {e.memify_metadata}")

async def show_node_weights(used_graph_element_ids, limit=6):
    """Print cognee's feedback weights for the graph nodes an answer leaned on."""
    from cognee.infrastructure.databases.graph import get_graph_engine
    node_ids = (used_graph_element_ids or {}).get("node_ids", [])
    g = await get_graph_engine()
    weights = await g.get_node_feedback_weights(node_ids)
    return {k: weights[k] for k in list(weights)[:limit]}

print("sample data:", sorted(p.name for p in DATA.glob("*.txt")))

## 1. Load the Slack data with `remember()`

Our Slack threads are saved as plain `.txt` transcripts in `sample_data/` — one
line per message, `[speaker, timestamp] text`. `remember()` takes raw text,
ingests it, and builds the knowledge graph in a single call.

> **In production** you wouldn't read files — you'd pull live threads from the
> Slack API and normalize them to this same shape (see
> `src/gtm_brain/sources/slack.py`). Once text is in this shape, the source
> doesn't matter.

In [ ]:
# Peek at the raw input first. Every source — Slack here, Granola later — is
# normalized to the same transcript shape: one line per message,
# "[speaker, timestamp] text". This is all cognee needs.
slack_files = sorted(DATA.glob("slack_*.txt"))
print(slack_files[1].read_text())   # slack_2: the node-sets bug report

In [ ]:
await reset()

slack_files = sorted(DATA.glob("slack_*.txt"))
for f in slack_files:
    result = await cognee.remember(f.read_text(), dataset_name=DATASET)
    print("remembered", f.name, "— status:", result.to_dict().get("status"))

## 2. Recall — ask the graph

`recall()` runs a graph-aware search and returns a natural-language answer by
default (it auto-routes the query for you). Pass `only_context=True` to get the
raw retrieved context instead of a synthesized answer — useful for seeing *what*
the graph pulled before it's summarized.

Our data: someone reported a bug about `remember` with node sets. Let's ask.

In [ ]:
results = await cognee.recall(
    "What bug was reported, and who reported it?",
    datasets=[DATASET],
)
print(answer(results))

In [ ]:
# Peek under the hood: the raw context the answer was built from
ctx = await cognee.recall(
    "node sets bug",
    datasets=[DATASET],
    only_context=True,
    top_k=3,
)
for c in ctx:
    print("-", str(getattr(c, "content", c))[:200])

## 3. Node sets — tag data so you can scope and permission it

**Node sets** are tags you attach at ingest time — `source:slack`, `project:x`,
`speaker:y`. They're how you later **scope** a query ("only answer from Slack")
and how you build **permissions** ("this user only sees these node sets").

You attach them by passing `node_set=[...]`, and you scope a recall with
`node_name=[...]`:

```python
await cognee.remember(text, dataset_name=ds, node_set=["source:slack"])
await cognee.recall(query, datasets=[ds], node_name=["source:slack"], scope=["graph"])
```

> **Heads-up for this demo:** scope **enforcement** is reliable on the Cognee
> server/cloud (where the bot in `BOT.md` runs). In this lightweight *in-process*
> notebook the recall-side filter is inconsistent, so don't be surprised if a
> scoped query here still pulls in everything. The tagging API below is exactly
> what you use in production.

In [ ]:
# Tag a couple of sources as we ingest them.
NS_DATASET = "node_sets_demo"
for f in slack_files:
    await cognee.remember(f.read_text(), dataset_name=NS_DATASET, node_set=["source:slack"])
for f in sorted(DATA.glob("granola*.txt")):
    await cognee.remember(f.read_text(), dataset_name=NS_DATASET, node_set=["source:granola"])
print("tagged Slack as source:slack and the meeting note as source:granola")

In [ ]:
# Scope a recall to one node set. (Reliable on the server; in-process this is
# the API shape rather than a hard guarantee — see the note above.)
scoped = await cognee.recall(
    "What was discussed about node sets?",
    datasets=[NS_DATASET],
    node_name=["source:slack"],
    scope=["graph"],
)
print(answer(scoped))

## 4. Custom graph model — extract a typed schema

Default extraction discovers entity types on its own, which can be noisy. When
you know the shape of your domain, give Cognee a **graph model**: a set of
Pydantic `DataPoint` classes. Extraction is then constrained to *your* types —
`Person`, `Message`, `Topic`, `Issue` — with the relationships you defined. Pair
it with a `custom_prompt` that tells the LLM how to fill the schema.

`remember()` accepts both, so this stays a one-call flow. (This is also the case
where you might instead split into `add()` + `cognify(graph_model=...)` if you
wanted to re-process existing data with a new schema without re-uploading it.)

Watch the two graphs below: the **before** is what default extraction
produced (lots of generic, self-discovered types), and the **after** is the
same conversations collapsed onto your `Person` / `Message` / `Issue` schema.

In [ ]:
# BEFORE: the graph from default extraction (generic, self-discovered types)
await show_graph("graph_default.html")

In [ ]:
from typing import Optional
from cognee.low_level import DataPoint

class Person(DataPoint):
    email: str
    name: str
    metadata: dict = {"index_fields": ["email", "name"], "identity_fields": ["email"]}

class Topic(DataPoint):
    label: str
    metadata: dict = {"index_fields": ["label"]}

class Issue(DataPoint):
    summary: str
    reported_by: Person
    metadata: dict = {"index_fields": ["summary"]}

class Message(DataPoint):
    speaker: Person
    text: str
    sent_at: str
    about: Optional[list[Topic]] = None
    reports: Optional[list[Issue]] = None
    metadata: dict = {"index_fields": ["text"]}

class Thread(DataPoint):
    channel: str
    started_at: str
    participants: list[Person]
    messages: list[Message]
    metadata: dict = {"index_fields": ["channel"]}

In [ ]:
EXTRACTION_PROMPT = """Extract a company conversation graph from the transcript.
Use the provided Thread graph model with Person, Message, Topic, and Issue nodes.
- Keep every transcript line as a Message with its exact speaker, timestamp, and text.
- Create one Person per email and reuse the same Person across messages.
- If a message reports a bug or broken behavior, create an Issue linked from that Message.
Prefer exact source text over summaries."""

await reset()
for f in slack_files:
    text = f.read_text()
    await cognee.remember(
        text,
        dataset_name=DATASET,
        node_set=["source:slack", f"speaker:{first_speaker(text)}"],
        graph_model=Thread,            # <-- typed schema
        custom_prompt=EXTRACTION_PROMPT,
    )
print("typed graph built")

In [ ]:
# AFTER: the same data, now constrained to the typed Person/Message/Issue schema
await show_graph("graph_typed.html")

In [ ]:
typed = await cognee.recall(
    "Summarize the reported issue and who is involved.",
    datasets=[DATASET],
)
print(answer(typed))

## 5. Feedback & self-improvement — and what changes in each store

A company brain should *learn*. Cognee improves in two ways:

- **Implicit** — `remember(..., self_improvement=True)` (the default) refines
  the graph on every write. You've had this in every cell above.
- **Explicit feedback loop** — `recall` → `add_feedback` → `improve` → `recall`.
  This teaches the brain things it can't extract from text, like *who owns what*.

We'll run that loop and **watch both stores change** at each step:

| Store | Inspect with | What `improve()` does to it |
|---|---|---|
| Session memory | `get_session()` | records your feedback, then flips `feedback_weights_applied` to `True` |
| Graph | `get_node_feedback_weights()` | shifts the feedback weight on the nodes the answer used |

Feedback needs session memory, so the setup cell sets `CACHING=true` and we pass
a `session_id` to `recall`. Scores are **1 (bad) … 5 (great)**.

In [ ]:
# BEAT 0 — a small, self-contained graph for this lesson, then snapshot BOTH stores.
# (Default extraction in its own dataset, so the answer is concrete and node weights
#  start at the default 0.5 — independent of the typed graph above.)
FB_DATASET = "feedback_demo"
SESSION = "feedback-demo"
for f in slack_files:
    await cognee.remember(f.read_text(), dataset_name=FB_DATASET)

Q = "Who is the go-to expert for node set problems?"
before = await cognee.recall(Q, datasets=[FB_DATASET], session_id=SESSION)
print("ANSWER (before):", answer(before))

qa = (await get_session(SESSION))[-1]          # the Q&A we just created
used = qa.used_graph_element_ids               # graph nodes/edges this answer used

print("\n— SESSION MEMORY (fresh turn) —");      await show_session(SESSION)
print("\n— GRAPH weights BEFORE —", await show_node_weights(used))

In [ ]:
# BEAT 1 — a teammate corrects it. Only SESSION MEMORY changes here.
await add_feedback(
    SESSION, qa.qa_id,
    feedback_text="Wrong owner. Milenko is the expert for node sets — route these to him.",
    feedback_score=1,                           # 1 = bad answer (scale is 1..5)
)
print("— SESSION MEMORY (feedback recorded, not yet applied) —")
await show_session(SESSION)
# note: memify_metadata is now {'feedback_weights_applied': False}

In [ ]:
# BEAT 2 — improve() folds the feedback in. Now BOTH stores change.
await cognee.improve(dataset=FB_DATASET, session_ids=[SESSION], feedback_alpha=0.8)

print("— SESSION MEMORY (applied) —");            await show_session(SESSION)
print("\n— GRAPH weights AFTER —", await show_node_weights(used))
# memify_metadata flips to True; the used nodes' weights drop 0.5 -> 0.1
# (they produced an answer the feedback rated low)

In [ ]:
# BEAT 3 — re-ask. The feedback now influences retrieval ranking.
after = await cognee.recall(Q, datasets=[FB_DATASET], session_id=SESSION, feedback_influence=1.0)
print("ANSWER (after):", answer(after))
# On a corpus this tiny the top answer may not move yet — but the weights that
# drive future ranking have changed, which is what you just saw in the graph store.

**What you just saw** — the proof is in the two stores, not the answer text:

- **Beat 1** touched *only* session memory: `feedback_text`/`feedback_score` got
  recorded, but `feedback_weights_applied` was still `False`.
- **Beat 2** is where `improve()` did its work: it flipped that flag to `True` and
  pushed the feedback into the **graph**, dropping the feedback weight on the nodes
  behind the low-rated answer (`0.5 → 0.1`). *That* is the improvement.

The recalled answer may not visibly change on a 3-message corpus (there's no
"Milenko is the expert" fact to surface, and one downvote can't flip a single
candidate). Over real usage these weights re-rank retrieval toward answers people
validated. Knobs: `feedback_score` (1–5), `feedback_alpha`, `feedback_influence`.

## 6. Add another source — Granola meeting notes

The payoff. The bug was *reported* in Slack, but it was *resolved* in a meeting
captured in a Granola note. Because Granola notes normalize to the same
transcript shape, adding them is the same `remember()` call with the same schema.
The note joins the **existing** graph, so a single question can span both sources.

In [ ]:
granola_files = sorted(DATA.glob("granola*.txt"))
for f in granola_files:
    await cognee.remember(
        f.read_text(),
        dataset_name=DATASET,
        node_set=["source:granola"],
        graph_model=Thread,
        custom_prompt=EXTRACTION_PROMPT,
    )
    print("remembered", f.name)

In [ ]:
# This answer requires connecting the Slack bug report to the Granola resolution
cross = await cognee.recall(
    "What was the fix for the node sets bug, and where was it explained?",
    datasets=[DATASET],
)
print(answer(cross))

# Node sets work across sources too — scope this query to just the Granola notes:
granola_now = await cognee.recall(
    "What bug was discussed?",
    datasets=[DATASET],
    node_name=["source:granola"],
    scope=["graph"],
)
print("scoped to source:granola ->", answer(granola_now))

## See the graph

`visualize_graph()` writes an interactive HTML view of the whole graph.

In [ ]:
await cognee.visualize_graph(str(ROOT / "graph.html"))
from IPython.display import IFrame
IFrame(src="graph.html", width="100%", height=600)

## 7. Merge the GTM brain — accounts, deals & calendar in one graph

Sections 1–6 turned *conversations* into a typed graph. The GTM brain adds the
*structured* side of go-to-market — the accounts, deals, signals, ICPs and
calendar around taking **GTM Tech Week from Warsaw 2026 → London 2027** — and
fuses both into one graph.

The merge works because of two identity anchors: **`Person.email`** and
**`Company.domain`**. Seed the structured nodes *first*, then cognify the
conversations on top — the extracted threads and messages attach onto the
existing Company/Person nodes instead of duplicating them.


In [ ]:
# Wire in the package pipeline (src/gtm_brain). It parses the vendored
# sample_data/gtm_brain/ folder: CSV tables, an .ics calendar, ICP docs,
# an account deep-dive, plus Granola transcripts and email threads.
import sys
SRC = ROOT / "src" if (ROOT / "src").exists() else ROOT.parent / "src"
sys.path.insert(0, str(SRC))

from gtm_brain.sources.gtm_files import GTMBrainSource
from gtm_brain.gtm_people import Roster

GTM_DIR = DATA / "gtm_brain"
gtm = GTMBrainSource(GTM_DIR, roster=Roster())

# Phase 1 build order matters: companies()/people() resolve the canonical
# nodes that deals/signals/calendar/threads then point at.
companies = gtm.companies()
people    = gtm.people()
events    = gtm.events()
deals     = gtm.deals()
signals   = gtm.signals()
calendar  = gtm.calendar_events()
icps      = gtm.icps()
docs      = list(gtm.conversation_docs())

print(f"companies={len(companies)} people={len(people)} deals={len(deals)} "
      f"signals={len(signals)} calendar={len(calendar)} icps={len(icps)} "
      f"conversations={len(docs)}")

clay = next(c for c in companies if c.domain == "clay.com")
print(f"\nanchor account: {clay.name} | tier {clay.tier} | icp_fit {clay.icp_fit_score} "
      f"| past_sponsor={clay.is_past_sponsor}")
print("a deal:", deals[0].name, "->", deals[0].company.domain, "@", deals[0].event.name)
print("a calendar invite:", calendar[0].summary, "->",
      [a.email for a in calendar[0].attendees])


**Seed structured nodes first, then cognify conversations.** This is the whole
trick — `seed_points()` persists Company/Person/Deal/... by their identity
fields, so `cognify` links onto them.


In [ ]:
from gtm_brain.cognee_client import seed_points, add_doc, cognify_threads

await reset()   # fresh graph for the merged demo

# Phase 1 — canonical structured nodes (no LLM)
await seed_points(companies + people + events)
await seed_points(deals + signals + calendar + icps)

# Phase 2 — stage each conversation, then cognify once into the Thread schema
for doc in docs:
    await add_doc(doc)
await cognify_threads()
print("merged graph built — conversations linked onto seeded accounts & people")


### Ask the merged graph

Each question below can only be answered by joining *across* sources — a deal
row, a meeting transcript, an email thread, a calendar invite and an account
deep-dive that all describe the same company.


In [ ]:
gtm_questions = [
    "What is the status of the Clay sponsorship and what did Bruno want instead of a booth?",
    "Can we pitch Pavilion a sponsorship, and what did Sam Jacobs agree to?",
    "Which Warsaw 2026 sponsors should we renew for London, and why?",
]
for q in gtm_questions:
    res = await cognee.recall(q, datasets=[DATASET])
    print("Q:", q)
    print(" ", answer(res))
    print()


### See the merged graph


In [ ]:
await show_graph("graph_gtm.html", height=600)


## Recap & where to go next

You built a GTM brain with one repeated call, then merged two graphs into one:

| Concept | Call |
|---|---|
| Load data + build graph | `cognee.remember(text, dataset_name=...)` |
| Ask questions | `cognee.recall(query, datasets=[...])` |
| Scope recall | `remember(..., node_set=[...])` + `recall(..., node_name=[...], scope=["graph"])` |
| Typed extraction | `remember(..., graph_model=Thread, custom_prompt=...)` |
| Improve / feedback | `add_feedback(...)` + `cognee.improve(dataset=..., feedback_alpha=...)` |
| New source | same `remember()` call, different input |
| Decouple ingest/build | drop to `cognee.add()` + `cognee.cognify(...)` |
| **Merge structured + conversations** | **`seed_points(nodes)` first, then `add_doc()` + `cognify_threads()`** |

The merge fuses on `Person.email` and `Company.domain`: seed the accounts,
deals, signals and calendar as canonical nodes, then cognify the conversations
so they attach onto them. One graph; ask it anything across sources.

From here, the full project (`src/gtm_brain/`) shows the product version:
live Slack/Granola ingestion, the GTM file source, an LLM classifier that
auto-tags threads, and recall-powered agents (account 360, pre-meeting briefs,
lookalike sponsors). See the repo `README.md` for hackathon tracks.


## Start Local UI And Backend

Start Cognee local UI from the imported `cognee` library. The backend uses this notebook kernel environment, so it sees the same local Cognee settings and database paths as the tutorial.


In [ ]:
import time
import urllib.request

import cognee

COGNEE_BACKEND_URL = "http://localhost:8000"
COGNEE_FRONTEND_URL = "http://localhost:3000"

cognee_ui_pids = []

def remember_cognee_ui_pid(pid_or_tuple):
    pid = pid_or_tuple[0] if isinstance(pid_or_tuple, tuple) else pid_or_tuple
    cognee_ui_pids.append(pid)
    print(f"Started Cognee UI process pid={pid}")

def url_ready(url: str, timeout_seconds: float = 1.5) -> bool:
    try:
        with urllib.request.urlopen(url, timeout=timeout_seconds) as response:
            return 200 <= response.status < 500
    except Exception:
        return False

def wait_for_url(name: str, url: str, timeout_seconds: int = 90) -> None:
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        if url_ready(url):
            print(f"{name} ready: {url}")
            return
        time.sleep(1)
    print(f"{name} did not answer within {timeout_seconds}s: {url}")

cognee_frontend_process = cognee.start_ui(
    pid_callback=remember_cognee_ui_pid,
    port=3000,
    open_browser=False,
    auto_download=True,
    start_backend=True,
    backend_port=8000,
    start_mcp=False,
)

if cognee_frontend_process is None:
    raise RuntimeError("Cognee UI did not start. Check the cell output above for npm/backend errors.")

wait_for_url("Cognee backend", f"{COGNEE_BACKEND_URL}/health")
wait_for_url("Cognee frontend", COGNEE_FRONTEND_URL, timeout_seconds=120)

print(f"Open the local UI: {COGNEE_FRONTEND_URL}")


## Terminate Local UI And Backend

Run the next cell when you are done.


In [ ]:
import os
import signal
import subprocess
import time

def process_is_running(pid: int) -> bool:
    try:
        os.kill(pid, 0)
        return True
    except OSError:
        return False

def stop_process_group(pid: int, timeout_seconds: int = 15) -> None:
    if not process_is_running(pid):
        print(f"Process pid={pid} is already stopped.")
        return

    print(f"Stopping Cognee UI process group pid={pid}")
    try:
        os.killpg(pid, signal.SIGTERM)
    except ProcessLookupError:
        print(f"Process group pid={pid} is already stopped.")
        return

    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        if not process_is_running(pid):
            print(f"Stopped pid={pid}.")
            return
        time.sleep(0.5)

    print(f"pid={pid} did not stop after SIGTERM; sending SIGKILL.")
    try:
        os.killpg(pid, signal.SIGKILL)
    except ProcessLookupError:
        pass

for pid in reversed(globals().get("cognee_ui_pids", [])):
    stop_process_group(pid)

cognee_ui_pids = []
cognee_frontend_process = None
